# BrainDecoding — Google Colab GPU 留出测试集 (`test`) 评估、负对照、子空间去偏与序列解码统一工作流

本 Notebook 统一整合了 **Google Colab GPU 实例** 上的**留出测试集 (`split=test`) 词级检索评估、时间错位/全零信号负对照、留一法共模扣除（CMS）、低秩子空间相干度门控去偏（Subspace LOO）与 Stage 3 大语言模型序列解码**：
- **支持三阶段完整横向对比（`--all-stages`）与单阶段独立评估**：
  - **阶段 1 (`word`)**：词级脑编码器（纯 CNN，`use_transformer: false`）
  - **阶段 2.1 (`context`)**：词级脑编码器 + 16 层全局上下文 Transformer（`use_transformer: true`，完整窗口，无局部滚动修订）
  - **阶段 2.2 (`revision`)**：词级脑编码器 + 16 层局部窗口滚动修订（`use_transformer: true`，`K0/K1/K2` 局部窗口 `[i-2, i+r]` 滚动回修）
  - **阶段 2.3 (`prefix_revision`)**：词级脑编码器 + 16 层全历史因果前缀滚动修订（`use_transformer: true`，`K0/K1/K2` 全历史因果前缀 `[1..i, i+r]` 滚动回修）
- **支持独立解耦的负对照与去偏配置体系（`configs/controls/`，单次 7x GPU 前向推理全部完成）**：
  - **时间错位真实信号负对照（Temporal Shift Control）**：在同受试者、同连续录音内对真实脑磁/脑电信号施加纯历史因果错位 $\delta \in \{-15\text{ s}, -30\text{ s}, -45\text{ s}, -60\text{ s}, -90\text{ s}, -120\text{ s}\}$ 确定性圆周移位，配置文件位于 `configs/controls/temporal_shift/`；
  - **留一法共模扣除（Leave-One-Out Common-Mode Subtraction）**：在单次前向推理完 1 个 Clean (0s) 与 6 个 Shifted 信号后，对 Clean (0s) 扣除全部 6 个 Shifted 均值（权重 $\eta \in \{0.5, 0.8, 1.0\}$），并对每个 Shifted 条件公平扣除其余 5 个 Shifted 均值，配置文件位于 `configs/controls/common_mode_subtraction/`；
  - **方案一：中心化低秩共模子空间投影 + 严格留一法基底去偏（Subspace Coherence LOO Debiasing）**：利用 6 个错位条件提取中心化 SVD 低秩偏置基底 $U_K$（每个错位条件严格仅用其余 5 个错位条件求留一法基底 $U_{K,\mathrm{LOO}}$，100% 保护正交的 $1024-K$ 个纯神经语义维度），配置文件位于 `configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml`；
  - **全零神经信号负对照（Zero-Signal Control）**：将输入脑信号置零以检验无脑信号时的机遇基线，配置文件位于 `configs/controls/zero_signal/`；
- **智能 45MB `.pt` 特征缓存与 3 秒子空间调参沙盒（Step 6B）**：
  - 任何包含 6 个时间错位窗口的评估（Step 5 或 Step 6）均会自动将归一化特征缓存至 `exploratory/subspace_encoded_cache_test.pt`（约 45MB）；
  - 后续在 **Step 6B** 调整低秩维度 $K$、扣除强度 $\eta$ 或门控参数时，**0 秒跳过 16.4GB 解包与神经网络前向推理**，3 秒直接产出完整 Markdown/CSV 对比表与四宫格图。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 选择要评估的数据集或单阶段 YAML 配置文件：
#   - 'configs/training/pallier2025_word_encoder.yaml'                              (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'                       (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'                          (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/pallier2025_prefix_revision.yaml'                           (法语 MEG 阶段 2.3: 词级脑编码器 + 16层全历史因果前缀滚动修订 [1..i, i+r])
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml' (法语 MEG 阶段 2.2: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_prefix_revision.yaml' (法语 MEG 阶段 2.3: 全历史因果前缀 + 孪生共模抵消 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_context.yaml'  (法语 MEG 阶段 2.1: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'                              (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'                       (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'                          (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/chineseeeg2_prefix_revision.yaml'                           (中文 EEG 阶段 2.3: 词级脑编码器 + 16层全历史因果前缀滚动修订 [1..i, i+r])
#   - 'configs/training/siamese_debiased/chineseeeg2_siamese_debiased_revision.yaml' (中文 EEG 阶段 2.2: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/chineseeeg2_siamese_debiased_prefix_revision.yaml' (中文 EEG 阶段 2.3: 全历史因果前缀 + 孪生共模抵消 + 预提取极速训练)
DATASET = 'pallier2025'               # 'pallier2025' 或 'chineseeeg2_littleprince'
CONFIG_FILE = 'configs/training/siamese_debiased/pallier2025_siamese_debiased_prefix_revision.yaml'
MODALITY = 'meg'                      # 'meg' (pallier2025) 或 'eeg' (chineseeeg2)

# 选择独立解耦的负对照与去偏配置文件（位于 configs/controls/，单次前向推理可同时完成标准评估 + 错位对照 + CMS + Subspace LOO）：
#   - 'configs/controls/temporal_shift/temporal_shift_control.yaml'                                (通用 -15s..-120s 纯历史因果时间错位负对照)
#   - 'configs/controls/common_mode_subtraction/common_mode_subtraction_control.yaml'              (留一法共模扣除 CMS: eta in {0.5, 0.8, 1.0} + 时间错位负对照)
#   - 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'    (方案一低秩子空间 LOO 去偏: K in {8,12,16,24}, eta in {0.8,1.0,1.2} + CMS + 时间错位)
#   - 'configs/controls/temporal_shift/pallier2025_temporal_shift_control.yaml'                     (Pallier2025 专属 -15s..-120s 纯历史因果时间错位负对照)
#   - 'configs/controls/temporal_shift/chineseeeg2_temporal_shift_control.yaml'                     (ChineseEEG-2 专属 -15s..-120s 纯历史因果时间错位负对照)
#   - 'configs/controls/zero_signal/zero_signal_control.yaml'                                      (全零脑磁/脑电信号负对照)
#   - ''                                                                                           (留空 '' 则仅运行标准 Clean 0s 评估)
CONTROL_CONFIG = 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'
SHIFT_SECONDS = '-15,-30,-45,-60,-90,-120'  # 纯历史因果时间错位窗口网格（秒；显式传参确保不依赖旧配置缓存，填 None 则从 YAML 读取）
# 显式指定多种子负对照汇总表（Negative Control Multi-Seed Summary）使用的去偏方法与主指标（支持单个或逗号/分号分隔同时输出多个方法与多个指标）：
#   CONTROL_METHOD 可选单个或多个：'auto' | 'raw' | 'zero_signal' | 'cms:eta=0.8' | 'cms:best_gap' | 'subspace:K=8,eta=0.8,gate=off' | 'subspace:best_gap' | 'subspace:best_clean' | 'all'
#     多方法示例：'raw,cms:eta=0.8,subspace:K=8,eta=0.8,gate=off,subspace:best_clean'
#   CONTROL_METRIC 可选单个或多个：'vocab50_macro_top10' | 'vocab50_macro_top1' | 'vocab50_micro_top10' | 'full_vocab_macro_top10' | 'full_vocab_macro_top1' | 'k2_vocab50_macro_top10' | 'k2_full_vocab_macro_top10' | 'all'
#     多指标示例：'vocab50_macro_top10,vocab50_macro_top1,full_vocab_macro_top10,k2_vocab50_macro_top10'
CONTROL_METHOD = 'raw,cms:eta=0.5'
CONTROL_METRIC = 'vocab50_macro_top10,vocab50_macro_top1,full_vocab_macro_top10,full_vocab_macro_top1'

EXP_TAG = '2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'              # 待评估的超参实验标签（定位 runs/<dataset>/<EXP_TAG>/<stage>/...；留空 '' 则为 runs/<dataset>/<stage>/...）
WORD_EXP_TAG = 'lr4e-4_bs256'                     # 若阶段 1 (word) 保存在另一个实验标签下，在此填入该标签名（留空 '' 默认与 EXP_TAG 相同；填 'default' 定位无标签根目录 word）
WINDOW_SECONDS = 1                 # 单字脑信号时间窗长度（秒，如 0.5, 0.75, 1.0, 1.25, 1.5, 2.0；填 None 则自动沿用检查点训练时的窗口长度）
WINDOW_START_OFFSET_SECONDS = None    # 相对于词起始时刻的窗口起点偏移（秒；填 None 则自动沿用检查点配置）
SEED = '0'                        # 随机种子：支持单种子（如 0）或逗号分隔的多种子列表（如 '0,1,2' 或 [0, 1, 2]）
SUBSET_ARGS = '--subject all'         # 必须与训练时的受试者子集保持一致
RUN_WORD_LEVEL_EVAL = True            # True：执行词级评估（Step 5 全阶段对比 或 Step 6 单阶段独立评估）；False：跳过词级评估
EVAL_ALL_STAGES = False               # False：仅单独评估 CONFIG_FILE 对应阶段（如阶段 1 word）；True：三阶段联合横向对比 (--all-stages)
RUN_SUBSPACE_LOO_SANDBOX = False      # True：执行 Step 6B 方案一低秩子空间 LOO 去偏 3 秒缓存沙盒或独立网格扫描
RUN_S3_SEQUENCE_DECODING = True       # True：执行 Step 7 Stage 3 (S3) 大语言模型序列解码（支持 SEED='0,1,2' 多种子连跑）

# 方案一（Subspace LOO）3 秒特征缓存调参沙盒参数（供下方 Step 6B 使用）：
SUBSPACE_STAGE = 'auto'               # 'auto' (自动跟随 CONFIG_FILE 阶段，支持 word / context / revision / prefix_revision) | 'prefix_revision' | 'revision' | 'context' | 'word' | 'all'
SUBSPACE_RANK_KS = '8,12,16,24'       # 低秩时间偏置子空间维度 K 网格
SUBSPACE_ETAS = '0.8,1.0,1.2'         # 子空间内扣除强度 eta 网格
SUBSPACE_COHERENCE_GATE = 'both'      # 'both' (同时对比 Gate=Off 与 Gate=On) | 'false' (仅 Gate=Off) | 'true'
SUBSPACE_COHERENCE_LOW = 0.35         # 相干度门控下界阈值 (理论 1/sqrt(6)=0.408)
SUBSPACE_COHERENCE_HIGH = 0.65        # 相干度门控上界阈值
SUBSPACE_GATE_MIN = 0.2               # 门控最小权重下限
SUBSPACE_GATE_MAX = 1.2               # 门控最大权重上限
SUBSPACE_CACHE_PT = ''                # 可选：显式指定某个 subspace_encoded_cache_test.pt 路径（留空 '' 则按 SEED_LIST 自动查找或编码）
SUBSPACE_FORCE_REENCODE = False       # True：强制忽略已缓存的 .pt 特征重新跑 GPU 编码

# Stage 3 (S3) 序列解码协议 v2 参数（供下方 Step 7 序列解码使用）：
#   - Colab L4 (22.5GB) / A100 (40GB+) 推荐使用 'Qwen/Qwen3-8B-Base'
#   - Colab T4 (15GB) 显存较紧时可切换为 'Qwen/Qwen3-4B-Base'
S3_REVISION_STAGE = 'auto'            # 'auto'（根据 CONFIG_FILE 自动识别）| 'prefix_revision' (Stage 2.3 全历史因果前缀) | 'revision' (Stage 2.2 局部窗口)
S3_LANGUAGE_MODEL = 'Qwen/Qwen3-8B-Base'
S3_TOP_K = 20                         # 闭集词表神经检索 Top-K 候选截断数（局部 Top-20 Softmax 归一化）
S3_BEAM_WIDTH = 5                     # 4 档输入延迟预算（S0 / K0 / K1 / K2 / F）统一自左向右束搜索束宽 B
S3_CMS_ETA = 0.0                      # 孪生去偏检查点推荐设为 0.0（或 0.25），保留 Clean 最高神经信噪比
S3_CONTROL_SHIFT = -75.0              # 训练未见的纯历史因果错位负对照秒数（-75s）
S3_K_OBSERVATIONS = '1,2,5,10'        # 跨被试多观测聚合曲线 k ∈ {1, 2, 5, 10}（同步覆盖 |V| ∈ {50, 100, 250, 2020}）
S3_VOCAB_SIZES = '50,100,250,500,1000,2020'  # 零训练成本多尺度候选词表扫描 |V|（|V|=2020 自动零开销复用已有全词库结果）
S3_CORE_VOCAB_SIZE = 250              # 主核心词表锚点 |V|=250（自动同时输出 |V| ∈ {50, 100, 250} 三档核心词表与 100%/80% 核心句集）
S3_SHARED_LANGUAGE_WEIGHT = False     # False（默认推荐）：状态自适应正下界单调递减 λ_K0 >= λ_K1 >= λ_K2 >= 0.05；True：K0/K1/K2 共享同一 λ >= 0.05
S3_FORCE_RERUN = False                # False：智能复用已有 .pt 矩阵并在检测到旧 λ_K2=0.0 时自动重校准重解码；True：强制全量重算
S3_MAX_SEGMENTS = 20                # 填 None 跑全量验证/测试集；填整数（如 20）可做极速联调冒烟
# ========================================================

_cfg_lower = str(CONFIG_FILE).lower()
if 'word_encoder' in _cfg_lower or _cfg_lower.endswith('main_word.yaml'):
    _INFERRED_CFG_STAGE = 'word'
elif 'prefix_revision' in _cfg_lower:
    _INFERRED_CFG_STAGE = 'prefix_revision'
elif 'context' in _cfg_lower:
    _INFERRED_CFG_STAGE = 'context'
else:
    _INFERRED_CFG_STAGE = 'revision'
if _INFERRED_CFG_STAGE == 'word' and (not EVAL_ALL_STAGES) and (not RUN_WORD_LEVEL_EVAL):
    print('[自动开启] 检测到 CONFIG_FILE 为阶段 1 (word) 词编码器配置，已自动设置 RUN_WORD_LEVEL_EVAL=True 以执行 Step 6 单阶段词评估！')
    RUN_WORD_LEVEL_EVAL = True

if isinstance(SEED, (list, tuple)):
    SEED_LIST = [int(s) for s in SEED]
else:
    SEED_LIST = [int(s.strip()) for s in str(SEED).replace(';', ',').split(',') if s.strip()]
SEED_ARG = ','.join(str(s) for s in SEED_LIST)

CONTROL_ARGS = f'--control-config {CONTROL_CONFIG}' if str(CONTROL_CONFIG).strip() else ''
if str(CONTROL_CONFIG).strip() and 'zero_signal' not in str(CONTROL_CONFIG).lower() and SHIFT_SECONDS is not None and str(SHIFT_SECONDS).strip() != '':
    CONTROL_ARGS = f'{CONTROL_ARGS} --shift-seconds {SHIFT_SECONDS}'.strip()
if CONTROL_METHOD is not None and str(CONTROL_METHOD).strip() != '':
    _cm_str = ','.join(str(x).strip() for x in CONTROL_METHOD) if isinstance(CONTROL_METHOD, (list, tuple)) else str(CONTROL_METHOD).strip()
    CONTROL_ARGS = f'{CONTROL_ARGS} --control-method "{_cm_str}"'.strip()
if CONTROL_METRIC is not None and str(CONTROL_METRIC).strip() != '':
    _cmt_str = ','.join(str(x).strip() for x in CONTROL_METRIC) if isinstance(CONTROL_METRIC, (list, tuple)) else str(CONTROL_METRIC).strip()
    CONTROL_ARGS = f'{CONTROL_ARGS} --control-metric "{_cmt_str}"'.strip()
WINDOW_ARGS = f'--window-seconds {WINDOW_SECONDS}' if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != '' else ''
if WINDOW_START_OFFSET_SECONDS is not None and str(WINDOW_START_OFFSET_SECONDS).strip() != '':
    WINDOW_ARGS = f'{WINDOW_ARGS} --window-start-offset-seconds {WINDOW_START_OFFSET_SECONDS}'.strip()
S3_MAX_SEG_ARGS = f'--max-segments {S3_MAX_SEGMENTS}' if S3_MAX_SEGMENTS is not None else ''
S3_EXTRA_ARGS = ' '.join(x for x in ('--shared-language-weight' if S3_SHARED_LANGUAGE_WEIGHT else '', '--force' if S3_FORCE_RERUN else '') if x)
SUBSPACE_REENCODE_ARGS = '--force-reencode' if SUBSPACE_FORCE_REENCODE else ''

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_ARCHIVE_DIR'] = DRIVE_ARCHIVE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('DATASET                   =', DATASET)
print('CONFIG_FILE               =', CONFIG_FILE)
print('SEEDS                     =', SEED_LIST, f'(--seed {SEED_ARG})')
print('RUN_WORD_LEVEL_EVAL       =', RUN_WORD_LEVEL_EVAL, f'(EVAL_ALL_STAGES={EVAL_ALL_STAGES})')
print('RUN_SUBSPACE_LOO_SANDBOX  =', RUN_SUBSPACE_LOO_SANDBOX, f'(STAGE={SUBSPACE_STAGE}, K={SUBSPACE_RANK_KS}, eta={SUBSPACE_ETAS})')
print('RUN_S3_SEQUENCE_DECODING  =', RUN_S3_SEQUENCE_DECODING)
print('CONTROL_CONFIG            =', CONTROL_CONFIG or '(disabled - clean only)')
print('SHIFT_SECONDS             =', SHIFT_SECONDS if (str(CONTROL_CONFIG).strip() and SHIFT_SECONDS) else '(from YAML / disabled)')
print('WINDOW_SECONDS            =', WINDOW_SECONDS if WINDOW_SECONDS is not None else '(auto from checkpoint / YAML)')
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])
print('BRAINDECODING_EXP_TAG     =', os.environ['BRAINDECODING_EXP_TAG'] or '(default)')
print('BRAINDECODING_WORD_EXP_TAG=', os.environ['BRAINDECODING_WORD_EXP_TAG'] or '(same as EXP_TAG / fallback default)')

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 初始化虚拟环境 + 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 秒级解包 Google Drive 上的 .tar 缓存归档回 /content/local_cache
#    若已存在预编码且与当前阶段/Full-Context/CMS_ETA 一致的 neural_vocabulary_scores_test.pt 或 subspace_encoded_cache_test.pt，则直接 0 秒跳过 16.4GB 解包！
import torch
_tag_part = EXP_TAG.strip() if (EXP_TAG and EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
_word_tag_part = WORD_EXP_TAG.strip() if (WORD_EXP_TAG and WORD_EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
_target_rev_stage = S3_REVISION_STAGE.strip() if str(S3_REVISION_STAGE).strip().lower() not in ('', 'auto', 'none') else (_INFERRED_CFG_STAGE if _INFERRED_CFG_STAGE in ('revision', 'prefix_revision') else 'prefix_revision')
def _is_valid_s3_pt(_pt_path: Path) -> bool:
    if not _pt_path.is_file():
        return False
    try:
        _payload = torch.load(_pt_path, map_location='cpu', weights_only=False)
        if not isinstance(_payload, dict):
            return False
        _cms = _payload.get('cms_metadata') or {}
        if abs(float(_cms.get('subtraction_eta', 0.0)) - float(S3_CMS_ETA)) > 1e-6:
            return False
        _clean = _payload.get('clean_scores') or {}
        if 'F' not in _clean:
            return False
        _prov = _payload.get('checkpoint_provenance') or {}
        if _prov.get('context') in (None, '', 'not_available'):
            return False
        _saved_stage = _prov.get('revision_stage')
        if _saved_stage is not None and str(_saved_stage) != _target_rev_stage:
            return False
        if _saved_stage is None and _target_rev_stage == 'prefix_revision':
            if 'prefix_revision' not in str(_prov.get('revision', '')).replace('\\', '/'):
                return False
        return True
    except Exception:
        return False

_all_exported_pts_exist = (not S3_FORCE_RERUN) and all(
    (_is_valid_s3_pt((Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part / 'sequence_decoding' / f'seed-{int(_s):03d}' / 'neural_vocabulary_scores_test.pt') if _tag_part else (Path(DRIVE_RESULTS_DIR) / DATASET / 'sequence_decoding' / f'seed-{int(_s):03d}' / 'neural_vocabulary_scores_test.pt'))
     and _is_valid_s3_pt((Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part / 'sequence_decoding' / f'seed-{int(_s):03d}' / 'neural_vocabulary_scores_val.pt') if _tag_part else (Path(DRIVE_RESULTS_DIR) / DATASET / 'sequence_decoding' / f'seed-{int(_s):03d}' / 'neural_vocabulary_scores_val.pt')))
    for _s in SEED_LIST
)
_subspace_stage_raw = str(SUBSPACE_STAGE).strip().lower() if SUBSPACE_STAGE is not None else 'auto'
if _subspace_stage_raw == 'all':
    _subspace_stages = ['context', 'revision']
elif _subspace_stage_raw in ('', 'auto', 'none'):
    _subspace_stages = [_INFERRED_CFG_STAGE]
else:
    _subspace_stages = [s.strip() for s in str(SUBSPACE_STAGE).split(',') if s.strip()]
def _has_subspace_pt(_st: str, _s: int) -> bool:
    _cand_roots = [(Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part / _st) if _tag_part else (Path(DRIVE_RESULTS_DIR) / DATASET / _st)]
    if _st == 'word' and _word_tag_part and _word_tag_part != _tag_part:
        _cand_roots.append(Path(DRIVE_RESULTS_DIR) / DATASET / _word_tag_part / _st)
    if _st == 'word':
        _cand_roots.append(Path(DRIVE_RESULTS_DIR) / DATASET / _st)
    return any(bool(list(_r.glob(f'*/seed-{int(_s):03d}/exploratory/subspace_encoded_cache_test.pt'))) for _r in _cand_roots)
_all_subspace_pts_exist = bool(str(SUBSPACE_CACHE_PT).strip() and Path(str(SUBSPACE_CACHE_PT).strip()).is_file()) or all(
    _has_subspace_pt(_st, _s)
    for _st in _subspace_stages
    for _s in SEED_LIST
)
if (not RUN_WORD_LEVEL_EVAL) and (not RUN_SUBSPACE_LOO_SANDBOX or (_all_subspace_pts_exist and not SUBSPACE_FORCE_REENCODE)) and (not RUN_S3_SEQUENCE_DECODING or _all_exported_pts_exist):
    if RUN_SUBSPACE_LOO_SANDBOX and _all_subspace_pts_exist:
        print(f'[0秒跳过解包] 已检测到云盘存在预编码特征缓存 subspace_encoded_cache_test.pt，直接进入 Step 6B 3秒低秩子空间去偏计算！')
    elif RUN_S3_SEQUENCE_DECODING and _all_exported_pts_exist:
        print(f'[0秒跳过解包] 已检测到云盘存在全部种子 {SEED_LIST} 的兼容 neural_vocabulary_scores_test.pt（阶段={_target_rev_stage}, 含 Full-Context），直接读取 .pt 矩阵，无需解包 16.4GB 原始脑信号缓存！')
    else:
        print('[0秒跳过解包] 仅读取已存 JSON 报表（Step 8），无需解包缓存！')
else:
    if RUN_S3_SEQUENCE_DECODING and not _all_exported_pts_exist:
        print(f'[提示] 云盘尚未缓存与当前阶段 ({_target_rev_stage}) / Full-Context / CMS_ETA={S3_CMS_ETA} 完全匹配的 .pt 得分矩阵，正在解包原始脑信号缓存以供重新导出...')
    !cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
        --archive-dir {DRIVE_ARCHIVE_DIR} \
        --cache-dir {LOCAL_CACHE_DIR} \
        --modality {MODALITY} \
        --skip-existing

In [ ]:
# 5. 【可选：RUN_WORD_LEVEL_EVAL=True 且 EVAL_ALL_STAGES=True 时执行】一键评估所有已训练阶段（word / context / revision，支持单种子或多种子 Mean ± SD 汇总）
if RUN_WORD_LEVEL_EVAL and EVAL_ALL_STAGES:
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
        --dataset {DATASET} \
        --seed {SEED_ARG} \
        --split test \
        --all-stages \
        {WINDOW_ARGS} \
        {CONTROL_ARGS} \
        {SUBSET_ARGS}
else:
    print(f'[跳过全阶段联合对比] RUN_WORD_LEVEL_EVAL={RUN_WORD_LEVEL_EVAL}, EVAL_ALL_STAGES={EVAL_ALL_STAGES}（将在下方 Step 6 单独评估 CONFIG_FILE={CONFIG_FILE}）。')

In [ ]:
# 6. 【RUN_WORD_LEVEL_EVAL=True 且 EVAL_ALL_STAGES=False 时执行】单独评估指定的 CONFIG_FILE（支持单种子或多种子 Mean ± SD 汇总，并自动落盘 45MB subspace_encoded_cache_test.pt）
if RUN_WORD_LEVEL_EVAL and (not EVAL_ALL_STAGES):
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
        --config {CONFIG_FILE} \
        --seed {SEED_ARG} \
        --split test \
        {WINDOW_ARGS} \
        {CONTROL_ARGS} \
        {SUBSET_ARGS}
else:
    print(f'[跳过单阶段独立评估] RUN_WORD_LEVEL_EVAL={RUN_WORD_LEVEL_EVAL}, EVAL_ALL_STAGES={EVAL_ALL_STAGES}。')

# from IPython.display import Image, display

# test_pngs = []
# for _s in SEED_LIST:
#     test_pngs.extend(Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(_s):03d}/figures/*test*.png'))
# test_pngs = sorted(set(test_pngs), key=lambda p: p.stat().st_mtime, reverse=True)
# allowed_tags = {t for t in (EXP_TAG, WORD_EXP_TAG) if t and t.lower() not in ('default', 'root', 'none')}
# shown = set()
# for png_path in test_pngs:
#     if allowed_tags and not any(t in png_path.parts for t in allowed_tags):
#         continue
#     key = (png_path.parent.parent.parent.parent.name, png_path.parent.parent.name, png_path.name)
#     if key in shown:
#         continue
#     shown.add(key)
#     print(f'\n[测试集图表] {png_path}')
#     display(Image(filename=str(png_path)))

In [ ]:
# 6B. 【可选：RUN_SUBSPACE_LOO_SANDBOX=True 时执行】方案一（中心化 SVD + 严格留一法基底 U_{K,LOO}）3秒缓存直读沙盒或独立网格扫描
#     若 Step 5/6 已运行过时间错位评估，此处会自动复用 exploratory/subspace_encoded_cache_test.pt 在 3 秒内完成全部 (K, eta, Gate) 网格扫描！
if RUN_SUBSPACE_LOO_SANDBOX:
    if str(SUBSPACE_CACHE_PT).strip():
        !cd {CODE_DIR} && {PYTHON} -u -m braindecoding.evaluation.subspace_coherence_debiasing_evaluation \
            --cache-pt {SUBSPACE_CACHE_PT} \
            --rank-ks {SUBSPACE_RANK_KS} \
            --etas {SUBSPACE_ETAS} \
            --coherence-gate {SUBSPACE_COHERENCE_GATE} \
            --coherence-low {SUBSPACE_COHERENCE_LOW} \
            --coherence-high {SUBSPACE_COHERENCE_HIGH} \
            --gate-min {SUBSPACE_GATE_MIN} \
            --gate-max {SUBSPACE_GATE_MAX}
    else:
        _subspace_ctrl = CONTROL_CONFIG if 'subspace_coherence_debiasing' in str(CONTROL_CONFIG) else 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'
        !cd {CODE_DIR} && {PYTHON} -u -m braindecoding.evaluation.subspace_coherence_debiasing_evaluation \
            --dataset {DATASET} \
            --stage {SUBSPACE_STAGE} \
            --config {CONFIG_FILE} \
            --control-config {_subspace_ctrl} \
            --seed {SEED_ARG} \
            --split test \
            --shift-seconds={SHIFT_SECONDS} \
            --rank-ks {SUBSPACE_RANK_KS} \
            --etas {SUBSPACE_ETAS} \
            --coherence-gate {SUBSPACE_COHERENCE_GATE} \
            --coherence-low {SUBSPACE_COHERENCE_LOW} \
            --coherence-high {SUBSPACE_COHERENCE_HIGH} \
            --gate-min {SUBSPACE_GATE_MIN} \
            --gate-max {SUBSPACE_GATE_MAX} \
            {WINDOW_ARGS} \
            {SUBSPACE_REENCODE_ARGS} \
            {SUBSET_ARGS}

    # from IPython.display import Image, display

    # subspace_pngs = []
    # for _s in SEED_LIST:
    #     subspace_pngs.extend(Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(_s):03d}/figures/subspace_coherence_debiasing_test.png'))
    # for png_path in sorted(set(subspace_pngs), key=lambda p: p.stat().st_mtime, reverse=True):
    #     if EXP_TAG and EXP_TAG.lower() not in ('', 'default', 'root', 'none') and EXP_TAG not in png_path.parts:
    #         continue
    #     print(f'\n[方案一低秩子空间去偏四宫格图] {png_path}')
    #     display(Image(filename=str(png_path)))
else:
    print('[跳过方案一沙盒] RUN_SUBSPACE_LOO_SANDBOX=False（提示：也可直接将 Cell 1 的 CONTROL_CONFIG 设为 subspace_coherence_debiasing_control.yaml 在 Step 5/6 一体化运行）。')

In [ ]:
# 7. 【可选：RUN_S3_SEQUENCE_DECODING=True 时执行】Stage 3 / S3 序列解码 + 多尺度候选词表 (|V|=50..2020) + 高覆盖核心集 (|V|=250)
if RUN_S3_SEQUENCE_DECODING:
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding decode-sentences \
        --step all \
        --training-config {CONFIG_FILE} \
        --revision-stage {S3_REVISION_STAGE} \
        --dataset {DATASET} \
        --split test \
        --seed {SEED_ARG} \
        --language-model {S3_LANGUAGE_MODEL} \
        --top-k {S3_TOP_K} \
        --beam-width {S3_BEAM_WIDTH} \
        --cms-eta {S3_CMS_ETA} \
        --control-shift-seconds {S3_CONTROL_SHIFT} \
        --k-observations {S3_K_OBSERVATIONS} \
        --vocab-sizes {S3_VOCAB_SIZES} \
        --core-vocab-size {S3_CORE_VOCAB_SIZE} \
        {S3_EXTRA_ARGS} \
        {WINDOW_ARGS} \
        {S3_MAX_SEG_ARGS} \
        {SUBSET_ARGS}

    # from IPython.display import Image, display

    # s3_pngs = []
    # for _s in SEED_LIST:
    #     s3_pngs.extend(Path(DRIVE_RESULTS_DIR).rglob(f'sequence_decoding/seed-{int(_s):03d}/evaluation_test/*.png'))
    # s3_pngs = sorted(set(s3_pngs), key=lambda p: p.stat().st_mtime, reverse=True)
    # for png_path in s3_pngs:
    #     if EXP_TAG and EXP_TAG not in ('default', 'root', 'none') and EXP_TAG not in png_path.parts:
    #         continue
    #     print(f'\n[S3 序列解码图表] {png_path}')
    #     display(Image(filename=str(png_path)))
else:
    print('[跳过 S3 序列解码] RUN_S3_SEQUENCE_DECODING=False（如需运行阶段 3 大语言模型序列解码，请将 Cell 1 中 RUN_S3_SEQUENCE_DECODING 设为 True）。')

In [ ]:
# 8. 【0秒纯读取报表与独立重绘】无需重跑任何模型，直接从已存 JSON 打印多种子 Mean ± SD 汇总看板、单种子看板、CMS/Subspace LOO 看板，并支持 0 秒重绘图表
import json, sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
from braindecoding.evaluation.sequence_decoding_evaluation import print_sequence_decoding_report
from braindecoding.evaluation.evaluation_reporting import render_saved_evaluation_report
from braindecoding.evaluation.evaluation_figures import replot_from_saved_json

VERBOSE_REPORT = False  # False: 打印 <=86 列单屏核心决策看板；True: 打印完整拆解长表
REPLOT_FIGURES = False  # True: 直接从 JSON 0秒重新生成 PNG+PDF 图表

_tag_part = EXP_TAG.strip() if (EXP_TAG and EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
_word_tag_part = WORD_EXP_TAG.strip() if (WORD_EXP_TAG and WORD_EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
_candidate_tag_dirs = []
for _t in (_tag_part, _word_tag_part, ''):
    _d = (Path(DRIVE_RESULTS_DIR) / DATASET / _t) if _t else (Path(DRIVE_RESULTS_DIR) / DATASET)
    if _d not in _candidate_tag_dirs and _d.exists():
        _candidate_tag_dirs.append(_d)

# 8.1 优先打印多种子汇总看板（multi_seed_stage_comparison_test.json 或 multi_seed_evaluation_test.json）
for _base_dir in _candidate_tag_dirs:
    for _ms_json in sorted(_base_dir.rglob('multi_seed_*_test.json')):
        print(render_saved_evaluation_report(_ms_json, verbose=VERBOSE_REPORT, control_method=CONTROL_METHOD, control_metric=CONTROL_METRIC))

# 8.2 按种子遍历打印三阶段对比看板、单阶段词级评估看板、CMS/Subspace LOO 探索性看板或 S3 序列解码看板
for _s in SEED_LIST:
    for _base_dir in _candidate_tag_dirs:
        _stage_cmp_json = _base_dir / f'stage_comparison_seed-{int(_s):03d}_test.json'
        if _stage_cmp_json.is_file():
            print(render_saved_evaluation_report(_stage_cmp_json, verbose=VERBOSE_REPORT))
            if REPLOT_FIGURES:
                replot_from_saved_json(_stage_cmp_json)
        for _st in ('word', 'context', 'revision', 'prefix_revision'):
            for _eval_json in sorted(_base_dir.glob(f'{_st}/*/seed-{int(_s):03d}/evaluation_test.json')):
                if len(SEED_LIST) == 1 or VERBOSE_REPORT:
                    print(render_saved_evaluation_report(_eval_json, verbose=VERBOSE_REPORT))
                if REPLOT_FIGURES:
                    replot_from_saved_json(_eval_json)
            for _exp_name in ('common_mode_subtraction_test.json', 'subspace_coherence_debiasing_test.json'):
                for _exp_json in sorted(_base_dir.glob(f'{_st}/*/seed-{int(_s):03d}/exploratory/{_exp_name}')):
                    if len(SEED_LIST) == 1 or VERBOSE_REPORT:
                        print(render_saved_evaluation_report(_exp_json, verbose=VERBOSE_REPORT))
                    if REPLOT_FIGURES:
                        replot_from_saved_json(_exp_json)
    _s3_dir = (Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part / 'sequence_decoding' / f'seed-{int(_s):03d}') if _tag_part else (Path(DRIVE_RESULTS_DIR) / DATASET / 'sequence_decoding' / f'seed-{int(_s):03d}')
    _summary_json_path = _s3_dir / 's3_evaluation_summary_test.json'
    if not _summary_json_path.is_file():
        _summary_json_path = _s3_dir / 'evaluation_test' / 's3_evaluation_summary_test.json'
    if _summary_json_path.is_file():
        if len(SEED_LIST) == 1 or VERBOSE_REPORT:
            _rep_data = json.loads(_summary_json_path.read_text(encoding='utf-8'))
            _rep_data.setdefault('exp_tag', EXP_TAG)
            _rep_data.setdefault('seed', int(_s))
            _rep_data.setdefault('cms_eta', S3_CMS_ETA)
            print_sequence_decoding_report(_rep_data, verbose=VERBOSE_REPORT)
        if REPLOT_FIGURES:
            replot_from_saved_json(_summary_json_path)

In [ ]:
# 8B. 【0秒旧版报表一键升级补全与多方法/多指标刷新】
#     直接读取已保存的旧版 multi_seed_evaluation_test.json（或尚无外部报表的单种子目录），
#     自动从同目录 seed-*/evaluation_test.json 及 seed-*/exploratory/*.json 零开销补全全部新增指标（全词表 Macro Top-1、K0/K1/K2、转移矩阵、实虚词、CMS/Subspace LOO），
#     并支持同时指定多个 --control-method 与多个 --control-metric 立即刷新 .json、.txt、.md 和 tables/*.csv！
import sys
from pathlib import Path
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
from braindecoding.evaluation.evaluation_reporting import upgrade_and_refresh_saved_multi_seed_reports

# 可选：显式指定某个 multi_seed_evaluation_test.json 文件路径或阶段目录（留空 '' 则自动扫描当前 DATASET / EXP_TAG 下的全部评估目录）
UPGRADE_TARGET_PATH = ''
# 支持同时输出多个负对照去偏方法（逗号或分号分隔，或填 'all'）
UPGRADE_CONTROL_METHODS = CONTROL_METHOD  # 如: 'raw,cms:eta=0.8,subspace:K=8,eta=0.8,gate=off,subspace:best_clean'
# 支持同时输出多个负对照评估指标（逗号分隔，或填 'all'）
UPGRADE_CONTROL_METRICS = CONTROL_METRIC  # 如: 'vocab50_macro_top10,vocab50_macro_top1,full_vocab_macro_top10,k2_vocab50_macro_top10'
UPGRADE_VERBOSE = False                   # True: 终端直接打印完整宽表；False: 终端打印紧凑看板（完整宽表始终自动写入 .txt/.md/.csv）

_targets_to_upgrade = []
if str(UPGRADE_TARGET_PATH).strip():
    _targets_to_upgrade.append(Path(str(UPGRADE_TARGET_PATH).strip()))
else:
    _tag_part = EXP_TAG.strip() if (EXP_TAG and EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
    _word_tag_part = WORD_EXP_TAG.strip() if (WORD_EXP_TAG and WORD_EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
    _default_root = (Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part) if _tag_part else (Path(DRIVE_RESULTS_DIR) / DATASET)
    if _default_root.exists():
        _targets_to_upgrade.append(_default_root)
    if _word_tag_part and _word_tag_part != _tag_part:
        _word_root = Path(DRIVE_RESULTS_DIR) / DATASET / _word_tag_part / 'word'
        if _word_root.exists() and _word_root not in _targets_to_upgrade:
            _targets_to_upgrade.append(_word_root)
    elif str(WORD_EXP_TAG).strip().lower() in ('default', 'root', 'none') and _tag_part:
        _word_root = Path(DRIVE_RESULTS_DIR) / DATASET / 'word'
        if _word_root.exists() and _word_root not in _targets_to_upgrade:
            _targets_to_upgrade.append(_word_root)

for _t_path in _targets_to_upgrade:
    print(f'[0秒升级与刷新] 正在扫描并补全目录/文件: {_t_path} ...')
    _upgraded_list = upgrade_and_refresh_saved_multi_seed_reports(
        _t_path,
        split='test',
        control_method=UPGRADE_CONTROL_METHODS,
        control_metric=UPGRADE_CONTROL_METRICS,
        verbose=UPGRADE_VERBOSE,
        print_to_stdout=True,
    )
    print(f'[完成] 共升级并刷新了 {len(_upgraded_list)} 份外部汇总报表（.json / .txt / .md / tables/*.csv）。')

In [ ]:
# 最后一个单元格
import os
import time
from google.colab import drive, runtime

# 1. 刷新系统层面的文件写缓冲
os.sync()

# 2. 强制 Google Drive 将挂载目录中的所有缓存刷新并同步至云端
drive.flush_and_unmount()

# 3. 预留 5-10 秒缓冲时间，确保网络请求收尾
print("数据已全部同步至 Google Drive，正在释放运行时以停止扣费...")
time.sleep(10)

# 4. 释放实例
runtime.unassign()